In [ ]:
GITHUB_REPO = "https://github.com/zariqq/weather-forecast.git"  # <-- change this
PROJECT_DIR = "/kaggle/working/project"

PLACE = "Lidar(Tomsk)"
DB_PATH = "./data/data-challenge.db"   # auto-downloaded by download_data.py if missing

TRAIN_START, TRAIN_END = "2009-01-01", "2021-12-31"
EVAL_START, EVAL_END   = "2022-01-01", "2023-12-31"

CTX_LEN = 8
EMBED_DIM = 64
EPOCHS = 30

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU visible — set Settings > Accelerator to GPU T4 x2 or P100, then Factory Reset / rerun.")


## Clone

In [ ]:
import os, shutil

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

!git clone {GITHUB_REPO} {PROJECT_DIR}
%cd {PROJECT_DIR}
!ls -la


## Install dependencies

In [ ]:
!grep -vi "^torch" requirements.txt > /tmp/requirements_no_torch.txt || true
!cat /tmp/requirements_no_torch.txt
!pip install -q -r /tmp/requirements_no_torch.txt

In [ ]:
from download_data import ensure_db

DB_PATH = ensure_db(DB_PATH)
print("DB ready at:", DB_PATH)

In [ ]:
!python train.py \
  --db {DB_PATH} --place "{PLACE}" \
  --train-start {TRAIN_START} --train-end {TRAIN_END} \
  --eval-start {EVAL_START} --eval-end {EVAL_END} \
  --ctx-len {CTX_LEN} --embed-dim {EMBED_DIM} --epochs {EPOCHS} \
  --ckpt /kaggle/working/weather_jepa.pt \
  --metrics-out /kaggle/working/rmse_metrics.json


## Inspect results

In [ ]:
import json

with open("/kaggle/working/rmse_metrics.json") as f:
    metrics = json.load(f)

print("Mean RMSE by horizon (kg/kg):", metrics["rmse_mean_by_horizon"])
print("Embedding dim:", metrics["embed_dim"])
print("Eval range:", metrics["eval_range"])


## Embedding visualization

Produces the PCA / t-SNE / cosine-similarity plots for the latent-space slide.
Saves PNGs into `/kaggle/working/` so they show up as notebook outputs.

In [ ]:
!python visualize_embeddings.py \
  --db {DB_PATH} --place "{PLACE}" \
  --ckpt /kaggle/working/weather_jepa.pt \
  --eval-start {EVAL_START} --eval-end {EVAL_END} \
  --horizon 3 \
  --out-prefix /kaggle/working/embeddings

from IPython.display import Image, display
display(Image("/kaggle/working/embeddings_pca.png"))
display(Image("/kaggle/working/embeddings_tsne.png"))
display(Image("/kaggle/working/embeddings_cosine_sim.png"))
